In [1]:
import sys
if not sys.warnoptions:
    import warnings
    warnings.simplefilter("ignore")

import pandas as pd
import datetime as dt
import scipy
import os


activity_map = {0 :"None",1:"walking",2:"sit",3:"stand",4:"sit-to-stand",5:"stand-to-sit",6:"turn-right",
                7:"turn-left"}
fog_map = {0:"no",1:"fog",2:"fog", 3:"fog"} 

data = pd.read_csv("final_dataset.csv")
data['activity-Torino'] = [activity_map[i] for i in data['activity-Torino']]
data['fog-Agree'] = [fog_map[i] for i in data['fog-Agree']]
PatientsData = data[['patient','session','task']].drop_duplicates().reset_index(drop=True)

In [2]:
def SGF(mov, column, f_low = 0.5, f_high = 20, fs = 60, order=4): 
    import numpy as np
    from scipy import signal
    
    nyquist_freq = fs / 2
    fc_low = f_low / nyquist_freq
    fc_high = f_high / nyquist_freq
    b, a = signal.butter(order, [fc_low, fc_high], btype='bandpass', output='ba')
    
    sgf_sign = signal.lfilter(b, a, mov[column])
    
    sgf_sign = sgf_sign - sgf_sign.mean()
    return sgf_sign  

In [3]:
def pre_fog_labeller(mov, fs = 60, colname = "fog-Agree", time_in_prefog = 1):
    fog = mov[colname]
    
    prefog_window = fs * time_in_prefog
    prefog_window = int(prefog_window)
    
    pos_transitions = [] 
    for i in range(len(fog)-1): 
        if fog[i] == "no" and fog[i+1] == "fog":
            pos_transitions.append(i+1) 

    if len(pos_transitions) == 0:
        return fog

    
    else:
        pos_transitions = [[i-prefog_window, i] for i in pos_transitions]
        pos_transitions = [[0,i[1]] if i[0] < 0 else i for i in pos_transitions]

        for pos in pos_transitions: 
            n = pos[1]-1 
            while n > pos[0]: 
                if fog[n] == "fog"or n == 0: 
                    break 
                n -= 1
            
            fog[n:pos[1]] = ['pre-fog'] * len(fog[n:pos[1]])
        
        return fog





In [4]:
def fog_window_detector_mode(mov, window_size = 60, overlap = 0, colname = "fog-Agree"): 
    fog = mov[colname]
    il = []
    step = int(round(window_size)) if overlap == 0 else int(round(window_size * (1 - overlap)))
    if len(fog) % window_size == 0 and overlap == 0:
        for i in range(0, len(fog), step):
            mode = pd.Series(fog[i:i+window_size]).mode()
            il.append(mode[0])
    else:
        for i in range(0, len(fog) - window_size + 1, step):
            mode = pd.Series(fog[i:i+window_size]).mode()
            il.append(mode[0])
    return il
    
def fog_window_detector_greedy(mov, window_size = 60, overlap = 0, colname = "fog-Agree"):
    fog = mov[colname]
    il = []
    step = int(round(window_size)) if overlap == 0 else int(round(window_size * (1 - overlap)))
    if len(fog) % window_size == 0 and overlap == 0:
        for i in range(0, len(fog), step):
            il.append(fog[i:i+window_size].iat[0])
    else:
        for i in range(0, len(fog) - window_size + 1, step):
            il.append(fog[i:i+window_size].iat[0])
    return il

In [5]:
def fog_interval_skip(mov, seconds_fog_to_skip, fog_colname = 'fog-Agree'): 
    
    fogwindowcurrent = mov.at[0, fog_colname]
    listoffogwindows = [[mov.at[0, fog_colname],0]]

    for i in range(1,len(mov.index)-1):
        if mov.at[i,fog_colname] != fogwindowcurrent:
            fogwindowcurrent = mov.at[i,fog_colname]
            listoffogwindows.append([mov.at[i,fog_colname],i])   
    
    fog_skipped = []
    window_to_skip_interval = seconds_fog_to_skip * 60 
    if len(listoffogwindows) != 1:
        for i in range(len(listoffogwindows)-1):
            if listoffogwindows[i][0] == 'no' and listoffogwindows[i+1][0] != 'no' and listoffogwindows[i][1] != 0: 
                window_eval = listoffogwindows[i+1][1] - listoffogwindows[i][1]
                if  window_eval <= window_to_skip_interval:
                    listoffogwindows[i][0] = listoffogwindows[i-1][0]
            
        for i in range(len(listoffogwindows)-1): 
            fog_skipped.extend([listoffogwindows[i][0]]*(listoffogwindows[i+1][1]-listoffogwindows[i][1]))
        fog_skipped.extend([listoffogwindows[-1][0]]*(len(mov.index)-listoffogwindows[-1][1]))
            
        return fog_skipped 
    
    else:
        return list(mov[fog_colname])

In [6]:
# This is a wrappper function that takes everything in one automatic step. 

# ---------------- Parameters -----------------------
# 
# data: signal original data, with labels and info on the sessions. 
#
# mode_fog_detector: "mode" or "greedy"- it choose which fog window labeller to use
#
# extraction_domain: 'all', 'statistical', 'temporal' or 'spectral' - TSFEL extractor domain configuration
#
# sensors_included: List - states whether to include all or a combination of sensors for the TSFEL step. 
#                   the sensors are ["ankle_left", 'ankle_right', 'lower_back', "wrist"] 
# 
# non_fog_interval_seconds_to_skip: number of seconds to skip if there's a interval between two fog events 
#
# PatientsData: dataframe implemented from the original that works as iterator. 
#
# fs: sampling rate of sensors
#
# window_size: lenght of windows for the segmentation of signal.
#
# overlap : overlap of the windows. 



def FoG_classification_prep(data,mode_fog_detector = "mode", label_pre_fog = False, time_in_prefog = 0, non_fog_interval_seconds_to_skip = 0, extraction_domain = 'all', PatientsData = PatientsData, fs = 60, window_size = 120, overlap = 0):         
    
    filename = "output_mode_{}_prefog_{}_ws_{}_ov_{}.csv".format(mode_fog_detector, time_in_prefog, window_size, overlap)
    
    for i in PatientsData.index:
        p = PatientsData.iloc[i,0]
        s = PatientsData.iloc[i,1]
        t = PatientsData.iloc[i,2]
        
        mov = data[(data['patient'] == p) & (data['session'] == s) & (data['task'] == t)].reset_index(drop=True)
        
        if non_fog_interval_seconds_to_skip != 0: 
            mov['fog-Agree'] = fog_interval_skip(mov, non_fog_interval_seconds_to_skip)

        if label_pre_fog == True:
            mov['fog-Agree'] = pre_fog_labeller(mov, fs, time_in_prefog = time_in_prefog)
        
        if not mov.isnull().values.any():
            
            for col in mov.columns[1:-8]:
                mov[col] = SGF(mov, col)
        
            if extraction_domain == "all": 
                extractor = tsfel.get_features_by_domain()
            elif extraction_domain == "statistical":
                extractor = tsfel.get_features_by_domain("statistical")
            elif extraction_domain == "temporal":
                extractor = tsfel.get_features_by_domain("temporal")
            elif extraction_domain == "spectral":
                extractor = tsfel.get_features_by_domain("spectral")
            
            res = tsfel.time_series_features_extractor(extractor, mov.iloc[:,1:-8], fs=fs, window_size=window_size, overlap=overlap)

            if mode_fog_detector == 'mode':
                res['label'] = fog_window_detector_mode(mov, window_size = window_size, overlap = overlap)
            else:
                res['label'] = fog_window_detector_greedy(mov, window_size = window_size, overlap = overlap)
            
            res["patient"] = p
            
            if os.path.exists(filename):
                res.to_csv(filename, header= False,  mode = "a")
            else:
                res.to_csv(filename)
        

In [ ]:
#os.mkdir("FeatureExtraction_prefogLab")
os.chdir("FeatureExtraction_prefogLab")

import tsfel
import itertools

skip = 0 
fs = 60

mode_window = ['mode',"greedy"]
window = [30,60,120,180]
prefog_time = [1,2,3,4,5]
overlap = [0,0.25,0.5,0.75] 
iterations = itertools.product(mode_window, window, prefog_time, overlap) 

for it in iterations:
    filename = "output_mode_{}_prefog_{}_ws_{}_ov_{}.csv".format(it[0], it[2], it[1], it[3])
    if os.path.exists(filename): 
        break
    else: 
        FoG_classification_prep(data=data, mode_fog_detector = it[0], label_pre_fog = True, time_in_prefog = it[2], non_fog_interval_seconds_to_skip = skip, extraction_domain = 'all', PatientsData = PatientsData, fs = fs, window_size = it[1], overlap = it[3]) 
os.chdir('..')      